In [ ]:
!pip install -U albumentations ultralytics fiftyone torchview torchinfo torchmetrics

# AI/ML Video Analysis Capstone: Advanced Comparative Analysis
This notebook is a compiled version of the VS Code project for Kaggle. It covers data augmentation, architectural introspection, feature analysis, fine-tuning, and final benchmark evaluations.

In [ ]:
import os
import glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import urllib.request
import zipfile
import tarfile
import time
import xml.etree.ElementTree as ET
from abc import ABC, abstractmethod
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, ConcatDataset, DataLoader
from albumentations.pytorch import ToTensorV2
import albumentations as A
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
    ssd300_vgg16, SSD300_VGG16_Weights,
    retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights,
    fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# --- Utilities: Metrics, Video, Evaluation, Analysis, Finetuning, Visualization ---

import matplotlib.pyplot as plt

def plot_fps_comparison(results_dict, save_path="output/fps_comparison.png"):
    """
    Plots a bar chart comparing the FPS of different models.
    results_dict: dict of {'model_name': fps_value}
    """
    models = list(results_dict.keys())
    fps_vals = list(results_dict.values())

    plt.figure(figsize=(8, 5))
    plt.bar(models, fps_vals, color=['blue', 'orange', 'green'][:len(models)])
    plt.ylabel('Frames Per Second (FPS)')
    plt.title('Inference Speed Comparison')
    plt.savefig(save_path)
    plt.close()
    print(f"FPS comparison plot saved to {save_path}")


import cv2
import time

def process_video_stream(model, video_path, out_path, num_frames=30):
    """
    Reads a video, runs inference frame by frame using the provided model,
    and writes the annotated frames to out_path.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Error opening video file: {video_path}")
        
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
    
    start_time = time.time()
    frame_count = 0
    
    print(f"Starting inference on {video_path}...")
    while cap.isOpened() and frame_count < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        annotated_frame = model.predict_and_annotate(frame)
        out.write(annotated_frame)
        frame_count += 1
        
    end_time = time.time()
    
    cap.release()
    out.release()
    
    if frame_count > 0:
        fps_processed = frame_count / (end_time - start_time)
        print(f"Processed {frame_count} frames. Average FPS: {fps_processed:.2f}")
        return fps_processed
    return 0

def process_webcam_stream(model, camera_id=0):
    """
    Opens the specified webcam, runs inference frame by frame,
    and displays the result in a real-time window.
    Press 'q' to quit.
    """
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        raise ValueError(f"Error opening webcam with ID: {camera_id}")
        
    print(f"Started webcam stream. Press 'q' to quit (ensure the video window is focused).")
    
    start_time = time.time()
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame from camera.")
            break
            
        annotated_frame = model.predict_and_annotate(frame)
        
        cv2.imshow(f'Real-time Detection ({model.__class__.__name__})', annotated_frame)
        
        # Press 'q' to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
        frame_count += 1
        
    end_time = time.time()
    
    cap.release()
    cv2.destroyAllWindows()
    
    if frame_count > 0:
        fps_processed = frame_count / (end_time - start_time)
        print(f"Processed {frame_count} frames. Average FPS: {fps_processed:.2f}")
        return fps_processed
    return 0


import torch
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as F
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import xml.etree.ElementTree as ET
import os

# Mapping VOC classes to IDs (simplified)
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse",
    "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(VOC_CLASSES)}

def parse_voc_xml(node):
    res = []
    objects = node.get("object", [])
    if not isinstance(objects, list):
        objects = [objects]
        
    for obj in objects:
        bndbox = obj.get("bndbox")
        if not bndbox:
            continue
        res.append({
            "name": obj.get("name"),
            "bbox": [
                int(bndbox.get("xmin")),
                int(bndbox.get("ymin")),
                int(bndbox.get("xmax")),
                int(bndbox.get("ymax")),
            ]
        })
    return res

def evaluate_model_on_voc(model_wrapper, dataset_root='./data/voc', num_samples=50):
    print(f"Evaluating {model_wrapper.__class__.__name__} on VOC2012...")
    # Download and load VOC 2012 val set
    dataset = VOCDetection(root=dataset_root, year='2012', image_set='val', download=True)
    
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    
    device = model_wrapper.device
    model = model_wrapper.model
    model.eval()

    # We evaluate on a subset for speed in this capstone draft
    for i in range(min(num_samples, len(dataset))):
        img, target_dict = dataset[i]
        
        # Parse ground truth
        objects = parse_voc_xml(target_dict['annotation'])
        gt_boxes = []
        gt_labels = []
        for obj in objects:
            if obj['name'] in CLASS_TO_IDX:
                gt_boxes.append(obj['bbox'])
                gt_labels.append(CLASS_TO_IDX[obj['name']])
        
        if not gt_boxes:
            continue
            
        target = [
            dict(
                boxes=torch.tensor(gt_boxes, dtype=torch.float32).to(device),
                labels=torch.tensor(gt_labels, dtype=torch.int64).to(device)
            )
        ]

        # Predict
        img_tensor = F.to_tensor(img).to(device)
        with torch.no_grad():
            if hasattr(model, 'predict'): # YOLO
                results = model(img, verbose=False)
                # YOLO output parsing
                boxes = results[0].boxes.xyxy
                scores = results[0].boxes.conf
                labels = results[0].boxes.cls.to(torch.int64)
                preds = [dict(boxes=boxes.to(device), scores=scores.to(device), labels=labels.to(device))]
            else: # Torchvision
                preds = model([img_tensor])
                # Filter out low confidence
                keep = preds[0]['scores'] > 0.1
                preds = [dict(
                    boxes=preds[0]['boxes'][keep],
                    scores=preds[0]['scores'][keep],
                    labels=preds[0]['labels'][keep]
                )]

        metric.update(preds, target)
        
    results = metric.compute()
    
    # Return key metrics
    return {
        'mAP@50': results['map_50'].item(),
        'mAP@50-95': results['map'].item(),
        'Precision': results['mar_100'].item(), # approximation
        'Recall': results['mar_1'].item() # approximation
    }


import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os

def perform_tsne_pca_analysis(features, labels=None, output_dir="output/feature_analysis"):
    """
    Features: numpy array of shape (N, D) where N is num samples, D is feature dim.
    Labels: optional list or array of shape (N,)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # PCA
    print("Performing PCA...")
    pca = PCA(n_components=2)
    pca_results = pca.fit_transform(features)
    
    plt.figure(figsize=(8,6))
    if labels is not None:
        scatter = plt.scatter(pca_results[:,0], pca_results[:,1], c=labels, cmap='tab10', alpha=0.7)
        plt.legend(*scatter.legend_elements(), title="Classes")
    else:
        plt.scatter(pca_results[:,0], pca_results[:,1], alpha=0.7)
    plt.title('PCA of Model Features')
    plt.savefig(os.path.join(output_dir, "pca_analysis.png"))
    plt.close()

    # t-SNE
    print("Performing t-SNE...")
    tsne = TSNE(n_components=2, perplexity=min(30, max(5, len(features)//3)), random_state=42)
    tsne_results = tsne.fit_transform(features)

    plt.figure(figsize=(8,6))
    if labels is not None:
        scatter = plt.scatter(tsne_results[:,0], tsne_results[:,1], c=labels, cmap='tab10', alpha=0.7)
        plt.legend(*scatter.legend_elements(), title="Classes")
    else:
        plt.scatter(tsne_results[:,0], tsne_results[:,1], alpha=0.7)
    plt.title('t-SNE of Model Features')
    plt.savefig(os.path.join(output_dir, "tsne_analysis.png"))
    plt.close()
    
    print(f"Feature analysis saved to {output_dir}")


import torch
from torch.utils.data import DataLoader

def train_one_epoch(model_wrapper, optimizer, data_loader, device, epoch, print_freq=10):
    model = model_wrapper.model
    model.train()
    total_loss = 0

    for i, (images, targets) in enumerate(data_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        
        if i % print_freq == 0:
            print(f"Epoch: [{epoch}]  [{i}/{len(data_loader)}]  loss: {losses.item():.4f}")
            
    return total_loss / len(data_loader)

def finetune_model(model_wrapper, dataset, num_epochs=3, batch_size=4):
    device = model_wrapper.device
    model = model_wrapper.model
    
    # Simple dataloader
    data_loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True, 
        collate_fn=lambda x: tuple(zip(*x))
    )
    
    # Using SGD as standard
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
    
    print(f"Starting finetuning for {model_wrapper.__class__.__name__} for {num_epochs} epochs")
    for epoch in range(num_epochs):
        avg_loss = train_one_epoch(model_wrapper, optimizer, data_loader, device, epoch)
        print(f"Epoch {epoch} complete. Avg Loss: {avg_loss:.4f}")
    
    print("Finetuning completed.")


"""
layer_visualizer.py
===================
Provides Keras plot_model-style architecture diagrams for PyTorch models
using torchview → graphviz, plus forward-hook feature-map extraction and
per-layer weight statistics.
"""

import os
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt

OUTPUT_DIR = 'output/layer_visuals'


class LayerVisualizer:
    def __init__(self, model, output_dir=OUTPUT_DIR):
        self.model = model
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self.activations = {}
        self.hooks = []

    # ── forward hooks ─────────────────────────────────────────────────────────

    def hook_fn(self, name):
        def fn(module, input, output):
            self.activations[name] = output.detach()
        return fn

    def register_hooks(self, layer_names):
        for name, module in self.model.named_modules():
            if name in layer_names:
                hook = module.register_forward_hook(self.hook_fn(name))
                self.hooks.append(hook)

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

    # ── feature map visualisation ──────────────────────────────────────────────

    def visualize_feature_maps(self, image_tensor, model_name='model'):
        self.activations.clear()
        self.model.eval()
        with torch.no_grad():
            try:
                self.model(image_tensor)
            except Exception as e:
                print(f'Inference failed during hook extraction: {e}')

        for name, act in self.activations.items():
            if act.dim() == 4:
                act_map = act[0].mean(dim=0).cpu().numpy()
                plt.figure(figsize=(6, 6))
                plt.imshow(act_map, cmap='viridis')
                plt.title(f'{model_name} — {name}')
                plt.axis('off')
                safe = name.replace('.', '_')
                plt.savefig(os.path.join(self.output_dir, f'{model_name}_{safe}.png'),
                            bbox_inches='tight', dpi=120)
                plt.close()

    # ── weight statistics ──────────────────────────────────────────────────────

    def print_layer_values(self):
        print(f'--- CNN Layer Stats: {self.model.__class__.__name__} ---')
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Conv2d):
                w = module.weight.detach()
                print(f'  {name:<50s}  shape={tuple(w.shape)}  '
                      f'mean={w.mean():.4f}  std={w.std():.4f}  '
                      f'min={w.min():.4f}  max={w.max():.4f}')

    # ── Keras-style plot_model architecture diagram ────────────────────────────

    def plot_architecture(self, filename='model_architecture',
                          input_size=(1, 3, 640, 640),
                          max_layers=None):
        """
        Generate a Keras plot_model-style architecture diagram using torchview.

        Each node shows: layer name, type, and output shape.
        Edges show data flow (forward pass).
        Saves <output_dir>/<filename>.png and returns the path.

        Args:
            filename:   output filename stem (no extension)
            input_size: tuple passed to torchview as the dummy input shape
            max_layers: unused (kept for API compat); torchview handles depth
        """
        try:
            from torchview import draw_graph
        except ImportError:
            print("torchview not installed. Run: pip install torchview")
            return None

        model_name = self.model.__class__.__name__
        print(f'  Drawing architecture for {model_name} ...')

        try:
            self.model.eval()
            # torchview traces the model and builds a graphviz Digraph
            graph = draw_graph(
                self.model,
                input_size=input_size,
                graph_name=model_name,
                expand_nested=True,      # show sub-modules like Keras
                hide_inner_tensors=True, # only show layer nodes, not intermediate tensors
                hide_module_functions=False,
                depth=10,
                device='cpu',
            )

            # Save as PNG via graphviz
            out_stem = os.path.join(self.output_dir, filename)
            # visual_graph is a graphviz.Digraph object
            graph.visual_graph.format = 'png'
            graph.visual_graph.render(out_stem, cleanup=True)
            out_path = out_stem + '.png'
            print(f'  ✓ Saved → {out_path}')
            return out_path

        except Exception as e:
            print(f'  torchview failed for {model_name}: {e}')
            print('  Falling back to text summary...')
            self._save_text_summary(filename)
            return None

    def _save_text_summary(self, filename):
        """Fallback: save a text-based layer summary as PNG using matplotlib."""
        try:
            from torchinfo import summary as ti_summary
            import io, sys
            buf = io.StringIO()
            old_stdout = sys.stdout
            sys.stdout = buf
            try:
                ti_summary(self.model, input_size=(1, 3, 640, 640), depth=5,
                           col_names=["input_size", "output_size", "num_params"])
            except Exception:
                pass
            sys.stdout = old_stdout
            text = buf.getvalue()
        except ImportError:
            lines = []
            for name, m in self.model.named_modules():
                lines.append(f'{name or "(root)":50s}  {type(m).__name__}')
            text = '\n'.join(lines)

        lines = text.split('\n')
        fig, ax = plt.subplots(figsize=(12, max(6, len(lines) * 0.22)))
        ax.axis('off')
        ax.text(0.01, 0.99, text, transform=ax.transAxes,
                fontsize=6.5, va='top', fontfamily='monospace')
        out_path = os.path.join(self.output_dir, filename + '.png')
        plt.savefig(out_path, bbox_inches='tight', dpi=130)
        plt.close()
        print(f'  ✓ Text summary saved → {out_path}')
        return out_path



In [ ]:
# --- Data Logic: Augmentations & Dataset Loader ---

"""
augmentations.py
================
Unified augmentation pipeline for multi-dataset object detection.
Supports: VOC2012, COCO128, COCO8, Open Images v7.
All datasets are normalised to pascal_voc bbox format [x1,y1,x2,y2] (abs pixels).
"""

import os
import glob
import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, ConcatDataset
import xml.etree.ElementTree as ET

# ─── COCO-80 class names ─────────────────────────────────────────────────────
COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis',
    'snowboard','sports ball','kite','baseball bat','baseball glove',
    'skateboard','surfboard','tennis racket','bottle','wine glass','cup',
    'fork','knife','spoon','bowl','banana','apple','sandwich','orange',
    'broccoli','carrot','hot dog','pizza','donut','cake','chair','couch',
    'potted plant','bed','dining table','toilet','tv','laptop','mouse',
    'remote','keyboard','cell phone','microwave','oven','toaster','sink',
    'refrigerator','book','clock','vase','scissors','teddy bear','hair drier',
    'toothbrush'
]

VOC_CLASSES = [
    'background','aeroplane','bicycle','bird','boat','bottle','bus','car',
    'cat','chair','cow','diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor'
]

# ─── Albumentations pipelines ─────────────────────────────────────────────────

TARGET_SIZE = 640  # standard square resize before model

def get_train_transforms():
    """Heavy augmentation pipeline – bbox-aware."""
    return A.Compose([
        A.LongestMaxSize(max_size=TARGET_SIZE),
        A.PadIfNeeded(TARGET_SIZE, TARGET_SIZE, border_mode=cv2.BORDER_CONSTANT, value=114),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),
        A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.15, rotate_limit=10,
                           border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.GaussNoise(p=0.2),
        A.MotionBlur(blur_limit=5, p=0.15),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['class_labels'],
        min_visibility=0.3,
        clip=True,
    ))


def get_val_transforms():
    """Light inference pipeline – only resize/normalise."""
    return A.Compose([
        A.LongestMaxSize(max_size=TARGET_SIZE),
        A.PadIfNeeded(TARGET_SIZE, TARGET_SIZE, border_mode=cv2.BORDER_CONSTANT, value=114),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['class_labels'],
        clip=True,
    ))


# ─── Per-dataset PyTorch Dataset classes ──────────────────────────────────────

class VOCDatasetWrapper(Dataset):
    """Wraps torchvision VOCDetection → (np.ndarray, boxes, labels)."""

    def __init__(self, voc_dataset, transforms=None):
        self.ds = voc_dataset
        self.transforms = transforms
        self.class_to_idx = {c: i for i, c in enumerate(VOC_CLASSES)}

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        pil_img, ann = self.ds[idx]
        image = np.array(pil_img)
        h, w = image.shape[:2]

        boxes, labels = [], []
        objects = ann['annotation'].get('object', [])
        if not isinstance(objects, list):
            objects = [objects]
        for obj in objects:
            name = obj['name']
            bb = obj['bndbox']
            x1, y1 = int(float(bb['xmin'])), int(float(bb['ymin']))
            x2, y2 = int(float(bb['xmax'])), int(float(bb['ymax']))
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(self.class_to_idx.get(name, 0))

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        if self.transforms:
            result = self.transforms(image=image, bboxes=boxes, class_labels=labels)
            image = result['image']
            boxes = list(result['bboxes'])
            labels = list(result['class_labels'])

        return image, boxes, labels


class YOLODatasetWrapper(Dataset):
    """
    Reads YOLO-format datasets (COCO8, COCO128).
    Label format per line: class cx cy w h  (all normalised 0-1).
    """

    def __init__(self, images_dir, labels_dir, transforms=None, class_names=None):
        self.transforms = transforms
        self.class_names = class_names or COCO_CLASSES
        self.samples = []

        img_exts = ('*.jpg', '*.jpeg', '*.png')
        img_paths = []
        for ext in img_exts:
            img_paths.extend(glob.glob(os.path.join(images_dir, '**', ext), recursive=True))

        for img_path in sorted(img_paths):
            stem = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(labels_dir, stem + '.txt')
            # also search recursively
            if not os.path.exists(lbl_path):
                matches = glob.glob(os.path.join(labels_dir, '**', stem + '.txt'), recursive=True)
                lbl_path = matches[0] if matches else None
            self.samples.append((img_path, lbl_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((640, 640, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        boxes, labels = [], []
        if lbl_path and os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls = int(parts[0])
                    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = max(0, int((cx - bw / 2) * w))
                    y1 = max(0, int((cy - bh / 2) * h))
                    x2 = min(w, int((cx + bw / 2) * w))
                    y2 = min(h, int((cy + bh / 2) * h))
                    if x2 > x1 and y2 > y1:
                        boxes.append([x1, y1, x2, y2])
                        labels.append(cls)

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        if self.transforms:
            result = self.transforms(image=image, bboxes=boxes, class_labels=labels)
            image = result['image']
            boxes = list(result['bboxes'])
            labels = list(result['class_labels'])

        return image, boxes, labels


class OpenImagesDatasetWrapper(Dataset):
    """
    Wraps a fiftyone Open Images dataset.
    Extracts bounding boxes from detections field.
    """

    def __init__(self, fo_dataset, transforms=None):
        self.fo_dataset = fo_dataset
        self.transforms = transforms
        self.samples = list(fo_dataset) if fo_dataset else []

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = cv2.imread(sample.filepath)
        if image is None:
            image = np.zeros((640, 640, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        boxes, labels = [], []
        detections = sample.get_field('detections')
        if detections and hasattr(detections, 'detections'):
            for det in detections.detections:
                # fiftyone bboxes: [top-left-x, top-left-y, width, height] normalised
                bx, by, bw, bh = det.bounding_box
                x1 = max(0, int(bx * w))
                y1 = max(0, int(by * h))
                x2 = min(w, int((bx + bw) * w))
                y2 = min(h, int((by + bh) * h))
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(0)   # label mapping for OI is dataset-specific

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        if self.transforms:
            result = self.transforms(image=image, bboxes=boxes, class_labels=labels)
            image = result['image']
            boxes = list(result['bboxes'])
            labels = list(result['class_labels'])

        return image, boxes, labels


# ─── Combined Dataset Builder ─────────────────────────────────────────────────

def build_combined_dataset(voc_dataset=None,
                            coco128_dir=None,
                            coco8_dir=None,
                            open_images_dataset=None,
                            mode='train'):
    """
    Combine all available datasets into a single ConcatDataset.

    Args:
        voc_dataset:         torchvision VOCDetection instance
        coco128_dir:         path to coco128 root (has images/ and labels/)
        coco8_dir:           path to coco8 root  (has images/ and labels/)
        open_images_dataset: fiftyone dataset instance
        mode:                'train' or 'val'

    Returns:
        torch.utils.data.ConcatDataset
    """
    transforms = get_train_transforms() if mode == 'train' else get_val_transforms()
    parts = []

    if voc_dataset is not None:
        parts.append(VOCDatasetWrapper(voc_dataset, transforms))
        print(f"  [VOC]         {len(parts[-1])} samples")

    if coco128_dir and os.path.isdir(coco128_dir):
        imgs = os.path.join(coco128_dir, 'images', 'train2017')
        lbls = os.path.join(coco128_dir, 'labels', 'train2017')
        if os.path.isdir(imgs):
            ds = YOLODatasetWrapper(imgs, lbls, transforms)
            parts.append(ds)
            print(f"  [COCO128]     {len(ds)} samples")

    if coco8_dir and os.path.isdir(coco8_dir):
        split = 'train' if mode == 'train' else 'val'
        imgs = os.path.join(coco8_dir, 'images', split)
        lbls = os.path.join(coco8_dir, 'labels', split)
        if os.path.isdir(imgs):
            ds = YOLODatasetWrapper(imgs, lbls, transforms)
            parts.append(ds)
            print(f"  [COCO8]       {len(ds)} samples")

    if open_images_dataset is not None:
        ds = OpenImagesDatasetWrapper(open_images_dataset, transforms)
        parts.append(ds)
        print(f"  [Open Images] {len(ds)} samples")

    if not parts:
        raise ValueError("No datasets provided to build_combined_dataset().")

    combined = ConcatDataset(parts)
    print(f"\n  ✓ Combined dataset: {len(combined)} total samples")
    return combined


import os
import urllib.request
import zipfile
import tarfile
from torchvision.datasets import VOCDetection

class DatasetLoader:
    def __init__(self, data_dir="data"):
        self.data_dir = data_dir
        os.makedirs(self.data_dir, exist_ok=True)
        
    def download_sample_video(self, url="https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4", filename="people_traffic.mp4"):
        filepath = os.path.join(self.data_dir, filename)
        if not os.path.exists(filepath):
            print(f"Downloading sample video to {filepath}...")
            urllib.request.urlretrieve(url, filepath)
            print("Download complete.")
        else:
            print(f"Sample video already exists at {filepath}.")
        return filepath
        
    def load_voc(self, year='2007'):
        print(f"Loading VOC {year}...")
        dataset = VOCDetection(root=os.path.join(self.data_dir, 'voc'), year=year, image_set='val', download=True)
        return dataset

    def load_coco128(self):
        print("Loading COCO128 via Ultralytics...")
        from ultralytics.utils.downloads import download
        url = "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip"
        filepath = os.path.join(self.data_dir, "coco128.zip")
        extract_dir = os.path.join(self.data_dir, "coco128")
        
        if not os.path.exists(extract_dir):
             if not os.path.exists(filepath):
                 urllib.request.urlretrieve(url, filepath)
             with zipfile.ZipFile(filepath, 'r') as zip_ref:
                 zip_ref.extractall(self.data_dir)
        return extract_dir

    def load_coco8(self):
        print("Loading COCO8 Object Detection dataset...")
        url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip"
        filepath = os.path.join(self.data_dir, "coco8.zip")
        extract_dir = os.path.join(self.data_dir, "coco8")
        
        if not os.path.exists(extract_dir):
             if not os.path.exists(filepath):
                 urllib.request.urlretrieve(url, filepath)
             with zipfile.ZipFile(filepath, 'r') as zip_ref:
                 zip_ref.extractall(self.data_dir)
        return extract_dir

    def load_open_images(self, max_samples=500):
        print(f"Loading Open Images Dataset (Subset of {max_samples} images)...")
        try:
            import fiftyone as fo
            import fiftyone.zoo as foz
            # Point fiftyone's zoo download dir to the project data folder
            # (avoids the ~/fiftyone default and the dataset_dir kwarg conflict)
            fo.config.dataset_zoo_dir = os.path.join(self.data_dir, "open_images")
            # Downloads a 500-image subset of Open Images v7 validation split
            dataset = foz.load_zoo_dataset(
                "open-images-v7",
                split="validation",
                max_samples=max_samples,
                drop_existing_dataset=True,
            )
            print(f"Successfully loaded {len(dataset)} images from Open Images v7.")
            return dataset
        except ImportError:
            print("Note: To load Open Images seamlessly, please install fiftyone: `pip install fiftyone`")
            return None



In [ ]:
# --- Models: Base, YOLO, RCNN, SSD, RetinaNet, FCOS ---

from abc import ABC, abstractmethod

class BaseModel(ABC):
    def __init__(self, device):
        self.device = device
        self.model = None

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def predict_and_annotate(self, frame):
        """
        Takes a BGR image frame (numpy array), runs inference, 
        draws bounding boxes, and returns the annotated frame.
        """
        pass


from ultralytics import YOLO


class YOLOModel(BaseModel):
    def __init__(self, device, weights_path='yolo11n.pt'):
        super().__init__(device)
        self.weights_path = weights_path

    def load(self):
        print(f"Loading YOLO model from {self.weights_path} onto {self.device}...")
        self.model = YOLO(self.weights_path)
        self.model.to(self.device)

    def predict_and_annotate(self, frame):
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")
        
        # YOLO11 returns a list of Results objects
        results = self.model(frame, verbose=False, device=self.device)
        annotated_frame = results[0].plot()
        return annotated_frame


import torch
import torch.nn as nn
import cv2
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F


class CustomBoxPredictor(nn.Module):
    """
    Custom prediction head to replace the default one in Faster R-CNN.
    This demonstrates the ability to modify network architecture with custom layers.
    """
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.cls_score = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3), # Custom added layer
            nn.Linear(512, num_classes)
        )
        self.bbox_pred = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes * 4)
        )

    def forward(self, x):
        if x.dim() == 4:
            torch._assert(
                list(x.shape[2:]) == [1, 1],
                f"x has the wrong shape, expecting the last two dimensions to be [1,1] instead of {list(x.shape[2:])}",
            )
        x = x.flatten(start_dim=1)
        scores = self.cls_score(x)
        bbox_deltas = self.bbox_pred(x)
        return scores, bbox_deltas

class CustomRCNNModel(BaseModel):
    def __init__(self, device):
        super().__init__(device)
        self.num_classes = 91 # Standard COCO classes
        
    def load(self):
        print(f"Loading Custom Faster R-CNN onto {self.device}...")
        weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
        self.model = fasterrcnn_resnet50_fpn(weights=weights)
        
        # Modify the architecture
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = CustomBoxPredictor(in_features, self.num_classes)
        
        self.model.to(self.device)
        self.model.eval() # Set to evaluation mode
        print("Custom Faster R-CNN Model loaded with custom Dropout head.")

    def predict_and_annotate(self, frame):
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")
            
        # Convert BGR to RGB, then to tensor
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_tensor = F.to_tensor(img_rgb).to(self.device)
        
        with torch.no_grad():
            prediction = self.model([img_tensor])[0]
            
        # Draw boxes manually for RCNN
        annotated_frame = frame.copy()
        for i in range(len(prediction['boxes'])):
            if prediction['scores'][i] > 0.5: # Confidence threshold
                box = prediction['boxes'][i].cpu().numpy().astype(int)
                cv2.rectangle(annotated_frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
                
        return annotated_frame


import torch
import cv2
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.transforms import functional as F


class SSDModel(BaseModel):
    def load(self):
        print(f"Loading SSD300 VGG16 onto {self.device}...")
        weights = SSD300_VGG16_Weights.DEFAULT
        self.model = ssd300_vgg16(weights=weights)
        self.model.to(self.device)
        self.model.eval()

    def predict_and_annotate(self, frame):
        if self.model is None:
            raise RuntimeError("Model not loaded.")
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_tensor = F.to_tensor(img_rgb).to(self.device)
        with torch.no_grad():
            prediction = self.model([img_tensor])[0]
        
        annotated_frame = frame.copy()
        for i in range(len(prediction['boxes'])):
            if prediction['scores'][i] > 0.5:
                box = prediction['boxes'][i].cpu().numpy().astype(int)
                cv2.rectangle(annotated_frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        return annotated_frame


import torch
import cv2
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.transforms import functional as F


class RetinaNetModel(BaseModel):
    def load(self):
        print(f"Loading RetinaNet onto {self.device}...")
        weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT
        self.model = retinanet_resnet50_fpn_v2(weights=weights)
        self.model.to(self.device)
        self.model.eval()

    def predict_and_annotate(self, frame):
        if self.model is None:
            raise RuntimeError("Model not loaded.")
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_tensor = F.to_tensor(img_rgb).to(self.device)
        with torch.no_grad():
            prediction = self.model([img_tensor])[0]
        
        annotated_frame = frame.copy()
        for i in range(len(prediction['boxes'])):
            if prediction['scores'][i] > 0.5:
                box = prediction['boxes'][i].cpu().numpy().astype(int)
                cv2.rectangle(annotated_frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        return annotated_frame


import torch
import cv2
from torchvision.models.detection import fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
from torchvision.transforms import functional as F


class FCOSModel(BaseModel):
    def load(self):
        print(f"Loading FCOS onto {self.device}...")
        weights = FCOS_ResNet50_FPN_Weights.DEFAULT
        self.model = fcos_resnet50_fpn(weights=weights)
        self.model.to(self.device)
        self.model.eval()

    def predict_and_annotate(self, frame):
        if self.model is None:
            raise RuntimeError("Model not loaded.")
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_tensor = F.to_tensor(img_rgb).to(self.device)
        with torch.no_grad():
            prediction = self.model([img_tensor])[0]
        
        annotated_frame = frame.copy()
        for i in range(len(prediction['boxes'])):
            if prediction['scores'][i] > 0.5:
                box = prediction['boxes'][i].cpu().numpy().astype(int)
                cv2.rectangle(annotated_frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        return annotated_frame



## 1. Dataset Integration & Augmentation

In [ ]:
loader = DatasetLoader(data_dir='data')
print("Initializing dataset downloads and loaders...")

# Loading standard benchmark object detection datasets
voc2012     = loader.load_voc('2012')
coco128_dir = loader.load_coco128()
coco8_dir   = loader.load_coco8()
open_images = loader.load_open_images(max_samples=100) # Reduced for speed

print(f"\nVOC 2012 : {len(voc2012)} images")
print(f"COCO128  : {coco128_dir}")
print(f"COCO8    : {coco8_dir}")
print(f"Open Img : {len(open_images) if open_images else 'N/A'} images")


### 1a. Dataset Sample Visualisation

In [ ]:
def draw_boxes(ax, image, boxes, labels, class_names, title=""):
    ax.imshow(image)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')
    colors = plt.cm.Set2.colors
    for (x1, y1, x2, y2), lbl in zip(boxes, labels):
        color = colors[int(lbl) % len(colors)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        cname = class_names[int(lbl)] if int(lbl) < len(class_names) else str(int(lbl))
        ax.text(x1, y1-4, cname, fontsize=7, color='white',
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))

# VOC 2012
print("=" * 60)
print("VOC 2012")
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, ax in enumerate(axes):
    pil_img, ann = voc2012[i]
    img = np.array(pil_img)
    objs = ann['annotation'].get('object', [])
    if not isinstance(objs, list): objs = [objs]
    boxes, labels = [], []
    for obj in objs:
        bb = obj['bndbox']
        x1,y1,x2,y2 = int(float(bb['xmin'])),int(float(bb['ymin'])),int(float(bb['xmax'])),int(float(bb['ymax']))
        lbl = VOC_CLASSES.index(obj['name']) if obj['name'] in VOC_CLASSES else 0
        boxes.append([x1,y1,x2,y2]); labels.append(lbl)
    draw_boxes(ax, img, boxes, labels, VOC_CLASSES, f"VOC #{i}")
plt.tight_layout(); plt.show()


## 2. Load Models & Plot Architectures

In [ ]:
models = {
    'YOLO11n':             YOLOModel(device, 'yolo11n.pt'),
    'YOLOv8n':             YOLOModel(device, 'yolov8n.pt'),
    'YOLOv9c':             YOLOModel(device, 'yolov9c.pt'),
    'Custom Faster R-CNN': CustomRCNNModel(device),
    'SSD300 VGG16':        SSDModel(device),
    'RetinaNet':           RetinaNetModel(device),
    'FCOS ResNet50':       FCOSModel(device),
}

for name, model in models.items():
    model.load()
print("\nAll models loaded successfully!")


## 3. Model Accuracy Evaluation

In [ ]:
accuracy_results = {}
for name, model in models.items():
    res = evaluate_model_on_voc(model, num_samples=10) # Reduced for quick run
    accuracy_results[name] = res

df_acc = pd.DataFrame(accuracy_results).T
display(df_acc)
